# Preprocessing

## Importing Libraries

In [5]:
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split
import time

IMG_DIR = "../data/raw/images_training_rev1/images_training_rev1"
LABELS_PATH = "../data/raw/training_solutions_rev1/training_solutions_rev1.csv"

In [ ]:
IMG_SIZE = 64

## Defining Class

In [2]:
labels_df = pd.read_csv(LABELS_PATH)
clean_df = labels_df[labels_df['Class1.3'] < 0.5].copy()

def assign_class(row):
    if row['Class1.1'] >= 0.5:
        return 'Smooth'
    elif row['Class4.1'] >= 0.5:
        return 'Spiral'
    else:
        return 'Irregular'

clean_df['label'] = clean_df.apply(assign_class, axis=1)
print(clean_df.shape)

(61534, 39)


## Testing Loading Function

### Testing loading on a Small Subset of 5 Images

In [ ]:
def load_and_resize(galaxy_id, img_dir=IMG_DIR, size=IMG_SIZE):
    path = os.path.join(img_dir, f"{galaxy_id}.jpg")
    img = Image.open(path).convert('RGB').resize((size, size))
    return np.array(img)

test_ids = clean_df['GalaxyID'].head(5).values
for gid in test_ids:
    arr = load_and_resize(gid)
    print(gid, arr.shape, arr.dtype)

100008 (64, 64, 3) uint8
100023 (64, 64, 3) uint8
100053 (64, 64, 3) uint8
100078 (64, 64, 3) uint8
100090 (64, 64, 3) uint8


### Testing loading on a Medium Subset of 2000 Images

In [4]:
subset_df = clean_df.sample(2000, random_state=0)

start = time.time()
X_subset = np.array([load_and_resize(gid) for gid in subset_df['GalaxyID']])
y_subset = subset_df['label'].values
elapsed = time.time() - start

print("Shape:", X_subset.shape)
print(f"Time for 2000 images: {elapsed:.2f} seconds")
print(f"Estimated time for full {len(clean_df)} images: {elapsed * len(clean_df) / 2000 / 60:.1f} minutes")

Shape: (2000, 64, 64, 3)
Time for 2000 images: 45.63 seconds
Estimated time for full 61534 images: 23.4 minutes


### Now Running it on the full Dataset (61k images)

In [5]:
start = time.time()
X = np.array([load_and_resize(gid) for gid in clean_df['GalaxyID']])
y = clean_df['label'].values
elapsed = time.time() - start

print("X shape:", X.shape)
print("y shape:", y.shape)
print(f"Total time: {elapsed/60:.1f} minutes")

X shape: (61534, 64, 64, 3)
y shape: (61534,)
Total time: 23.7 minutes


## Normalization - converting every value to be within 0 and 1

In [6]:
X_norm = X.astype('float32') / 255.0
print(X_norm.dtype, X_norm.min(), X_norm.max())

float32 0.0 1.0


## Labeling Class with numbers

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(le.classes_)
print(y_encoded[:10])

['Irregular' 'Smooth' 'Spiral']
[0 2 1 1 1 1 0 1 2 0]


##  train/val/test spliting

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_norm, y_encoded, test_size=0.3, random_state=0, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=0, stratify=y_temp
)
#stratify=y_encoded / y_temp, makes sure that when it splits, the class propotions are preserved in both the output groups, matching the original datasets propotions.

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (43073, 64, 64, 3) Val: (9230, 64, 64, 3) Test: (9231, 64, 64, 3)


## Saving processed Arrays

In [9]:
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_val.npy', X_val)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)
print("Saved.")

Saved.


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Part 2: Confidence Filtering

The original preprocessing (Part 1) used a 0.5 vote-fraction threshold to assign
labels — meaning if just over half of volunteers agreed on a class, that galaxy
got that label. This works as a starting point but has a real problem: a galaxy
where 51% of people said "Smooth" and 49% said "Irregular" gets the same clean
"Smooth" label as one where 95% of people agreed. The model has no way to
distinguish confident examples from genuinely ambiguous ones.

Confidence filtering raises that bar to 0.7 — only galaxies where at least 70%
of volunteers agreed on a class are kept. Galaxies that don't clear this threshold
for any class are dropped entirely, since they're genuinely ambiguous even to
human experts and including them as training examples adds noise rather than signal.

### What changed in the labeling logic

Original threshold (Part 1):
- Class1.1 >= 0.5 → Smooth
- Class4.1 >= 0.5 → Spiral
- Otherwise → Irregular
- Discard: Class1.3 >= 0.5 (artifacts)

Confidence filtered threshold (Part 2):
- Class1.1 >= 0.7 → Smooth
- Class4.1 >= 0.7 → Spiral
- Class1.2 >= 0.7 (and not spiral/edge-on) → Irregular
- Discard: anything that doesn't clearly meet one of the above thresholds

### Results of filtering

| | Original (0.5 threshold) | Confident (0.7 threshold) |
|---|---|---|
| Total galaxies | 61,534 | 36,812 |
| Smooth | 25,868 | 14,146 |
| Irregular | 25,269 | 16,153 |
| Spiral | 10,397 | 6,513 |
| Dropped | — | 24,766 (~40%) |

About 40% of the original dataset didn't have a clear majority vote at the 0.7
threshold — meaning nearly half the galaxies were genuinely ambiguous to human
volunteers. This is itself an interesting finding about the difficulty of galaxy
morphology classification, and explains why even good models struggle to push
past certain accuracy ceilings on this dataset.

Processed arrays saved with `_conf` suffix (e.g., `X_train_conf.npy`) to keep
them separate from the original split — both versions are available for comparison.

In [6]:
labels_df = pd.read_csv(LABELS_PATH)

def assign_class_confident(row):
    if row['Class1.1'] >= 0.7:
        return 'Smooth'
    elif row['Class1.3'] >= 0.7:
        return None  # artifact/star, discard
    elif row['Class4.1'] >= 0.7:
        return 'Spiral'
    elif row['Class1.2'] >= 0.7:
        return 'Irregular'
    else:
        return None  # ambiguous, discard

labels_df['label'] = labels_df.apply(assign_class_confident, axis=1)
confident_df = labels_df[labels_df['label'].notna()].copy()

print(f"Original dataset: {len(labels_df)}")
print(f"After confidence filtering (>=0.7): {len(confident_df)}")
print(f"Dropped: {len(labels_df) - len(confident_df)}")
print()
print(confident_df['label'].value_counts())

Original dataset: 61578
After confidence filtering (>=0.7): 36812
Dropped: 24766

label
Irregular    16153
Smooth       14146
Spiral        6513
Name: count, dtype: int64


## loading and resizing images for Confident Subset

In [7]:
import time
import numpy as np
from PIL import Image
import os

IMG_DIR = "../data/raw/images_training_rev1/images_training_rev1"
IMG_SIZE = 64

def load_and_resize(galaxy_id, img_dir=IMG_DIR, size=IMG_SIZE):
    path = os.path.join(img_dir, f"{galaxy_id}.jpg")
    img = Image.open(path).convert('RGB').resize((size, size))
    return np.array(img)

print(f"Loading {len(confident_df)} images...")
start = time.time()
X_conf = np.array([load_and_resize(gid) for gid in confident_df['GalaxyID'].values])
y_conf = confident_df['label'].values
elapsed = time.time() - start

print(f"X shape: {X_conf.shape}")
print(f"y shape: {y_conf.shape}")
print(f"Time: {elapsed/60:.1f} minutes")

Loading 36812 images...
X shape: (36812, 64, 64, 3)
y shape: (36812,)
Time: 17.8 minutes


## Normalization and Encoding

In [8]:
from sklearn.preprocessing import LabelEncoder

X_conf_norm = X_conf.astype('float32') / 255.0
print(f"dtype: {X_conf_norm.dtype}, min: {X_conf_norm.min():.2f}, max: {X_conf_norm.max():.2f}")

le_conf = LabelEncoder()
y_conf_encoded = le_conf.fit_transform(y_conf)
print(f"Classes: {le_conf.classes_}")
print(f"Encoded sample: {y_conf_encoded[:10]}")

dtype: float32, min: 0.00, max: 1.00
Classes: ['Irregular' 'Smooth' 'Spiral']
Encoded sample: [1 1 1 0 0 0 0 1 0 0]


## train/val/test splitting

In [9]:
from sklearn.model_selection import train_test_split

X_conf_train, X_conf_temp, y_conf_train, y_conf_temp = train_test_split(
    X_conf_norm, y_conf_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_conf_encoded
)

X_conf_val, X_conf_test, y_conf_val, y_conf_test = train_test_split(
    X_conf_temp, y_conf_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_conf_temp
)

print(f"Train: {X_conf_train.shape}")
print(f"Val:   {X_conf_val.shape}")
print(f"Test:  {X_conf_test.shape}")

Train: (25768, 64, 64, 3)
Val:   (5522, 64, 64, 3)
Test:  (5522, 64, 64, 3)


##  Saving to Separate Files

In [10]:
np.save('../data/processed/X_train_conf.npy', X_conf_train)
np.save('../data/processed/X_val_conf.npy', X_conf_val)
np.save('../data/processed/X_test_conf.npy', X_conf_test)
np.save('../data/processed/y_train_conf.npy', y_conf_train)
np.save('../data/processed/y_val_conf.npy', y_conf_val)
np.save('../data/processed/y_test_conf.npy', y_conf_test)
print("Saved confident split arrays.")
print(f"Label mapping: {dict(zip(le_conf.classes_, le_conf.transform(le_conf.classes_)))}")

Saved confident split arrays.
Label mapping: {'Irregular': np.int64(0), 'Smooth': np.int64(1), 'Spiral': np.int64(2)}
